# Step 3: Advanced Feature Engineering

Machine learning models cannot spot a crisis by looking at a single isolated data point.
We must explicitly encode spatial adjacency and temporal momentum into our dataset.

This notebook engineers:
1. **Temporal Lag Features** - lag_1_day, lag_2_day, lag_7_day, rolling_mean_3_day
2. **Spatial K-Ring Features** - neighbor_pressure_index from surrounding hexagons
3. **Out-of-Time Chronological Split** - 60/20/20 train/val/test to prevent data leakage

In [1]:
import polars as pl
import h3
import numpy as np
import datetime
import warnings
warnings.filterwarnings('ignore')

## 1. Load Zero-Padded Matrix from Step 2
We load the structurally complete grid matrix with every hexagon having an unbroken daily sequence.

In [2]:
df = pl.read_csv('aggregated_spatial_matrix.csv', try_parse_dates=True)
df = df.with_columns(pl.col('time_bucket_1D').cast(pl.Date))
df = df.rename({'time_bucket_1D': 'date_bucket', 'hex_id': 'h3_index'})
print(f'Loaded {df.height} rows across {df["h3_index"].n_unique()} unique hexagons.')
df.head()

Loaded 726660 rows across 4037 unique hexagons.


date_bucket,h3_index,event_count,peak_intensity,energy_output
date,str,i64,f64,f64
2025-12-14,"""8400405ffffffff""",0,0.0,0.0
2025-12-15,"""8400405ffffffff""",0,0.0,0.0
2025-12-16,"""8400405ffffffff""",0,0.0,0.0
2025-12-17,"""8400405ffffffff""",0,0.0,0.0
2025-12-18,"""8400405ffffffff""",0,0.0,0.0


## 2. Temporal Lag and Momentum Features

Temporal lags force the model to look backward in time.
Because date gaps are filled in Step 2, we can use Polars window functions
`.shift().over('h3_index')` to compute per-hexagon lags cleanly.

| Feature | Description |
|---|---|
| `lag_1_day` | Event count yesterday |
| `lag_2_day` | Event count 2 days ago |
| `lag_7_day` | Event count last week |
| `rolling_mean_3_day` | 3-day momentum average |

In [3]:
df = df.sort(['h3_index', 'date_bucket'])

df_lag = df.with_columns([
    pl.col('event_count').shift(1).over('h3_index').fill_null(0).alias('lag_1_day'),
    pl.col('event_count').shift(2).over('h3_index').fill_null(0).alias('lag_2_day'),
    pl.col('event_count').shift(7).over('h3_index').fill_null(0).alias('lag_7_day'),
    pl.col('event_count').rolling_mean(window_size=3, min_periods=1).over('h3_index').alias('rolling_mean_3_day')
])

print('Temporal lag features engineered successfully.')
df_lag.select(['date_bucket', 'h3_index', 'event_count', 'lag_1_day', 'lag_7_day', 'rolling_mean_3_day']).head(10)

Temporal lag features engineered successfully.


date_bucket,h3_index,event_count,lag_1_day,lag_7_day,rolling_mean_3_day
date,str,i64,i64,i64,f64
2025-12-14,"""8400405ffffffff""",0,0,0,0.0
2025-12-15,"""8400405ffffffff""",0,0,0,0.0
2025-12-16,"""8400405ffffffff""",0,0,0,0.0
2025-12-17,"""8400405ffffffff""",0,0,0,0.0
2025-12-18,"""8400405ffffffff""",0,0,0,0.0
2025-12-19,"""8400405ffffffff""",0,0,0,0.0
2025-12-20,"""8400405ffffffff""",0,0,0,0.0
2025-12-21,"""8400405ffffffff""",0,0,0,0.0
2025-12-22,"""8400405ffffffff""",0,0,0,0.0


## 3. Spatial Neighborhood Context (H3 K-Ring)

An outbreak does not stop at an imaginary hexagon border.
If a target hexagon has 0 events but all 6 surrounding neighbors spike to 100,
an active crisis is **encroaching**.

We use `h3.k_ring(hex_id, 1)` to get adjacent hexagons and compute a
**Neighbor Encroachment Index** (mean event count of all 6 neighbors).

In [4]:
unique_hex_list = df_lag['h3_index'].unique().to_list()
hex_neighbor_map = {}

for hex_id in unique_hex_list:
    try:
        if hasattr(h3, 'k_ring'):
            neighbors = list(h3.k_ring(hex_id, 1))
        else:
            neighbors = list(h3.grid_disk(hex_id, 1))
        hex_neighbor_map[hex_id] = [n for n in neighbors if n != hex_id]
    except Exception:
        hex_neighbor_map[hex_id] = []

print(f'Built neighbor map for {len(hex_neighbor_map)} unique hexagons.')

Built neighbor map for 4037 unique hexagons.


In [5]:
spatial_lookup = (
    df_lag.select(['date_bucket', 'h3_index', 'event_count'])
    .to_pandas()
    .set_index(['date_bucket', 'h3_index'])['event_count']
    .to_dict()
)
print(f'Spatial lookup table contains {len(spatial_lookup)} entries.')

Spatial lookup table contains 726660 entries.


In [6]:
neighbor_avgs = []
for row in df_lag.iter_rows(named=True):
    date = row['date_bucket']
    current_hex = row['h3_index']
    neighbor_ids = hex_neighbor_map.get(current_hex, [])
    neighbor_values = [spatial_lookup.get((date, nid), 0) for nid in neighbor_ids]
    neighbor_avgs.append(float(np.mean(neighbor_values)) if neighbor_values else 0.0)

final_engineered_dataset = df_lag.with_columns(
    pl.Series('neighbor_pressure_index', neighbor_avgs, dtype=pl.Float32)
)

print('Spatial K-Ring Neighbor Encroachment Index computed!')
final_engineered_dataset.select(['date_bucket', 'h3_index', 'event_count', 'lag_1_day', 'lag_7_day', 'rolling_mean_3_day', 'neighbor_pressure_index']).head(15)

Spatial K-Ring Neighbor Encroachment Index computed!


date_bucket,h3_index,event_count,lag_1_day,lag_7_day,rolling_mean_3_day,neighbor_pressure_index
date,str,i64,i64,i64,f64,f32
2025-12-14,"""8400405ffffffff""",0,0,0,0.0,0.0
2025-12-15,"""8400405ffffffff""",0,0,0,0.0,0.0
2025-12-16,"""8400405ffffffff""",0,0,0,0.0,0.0
2025-12-17,"""8400405ffffffff""",0,0,0,0.0,0.0
2025-12-18,"""8400405ffffffff""",0,0,0,0.0,0.0
…,…,…,…,…,…,…
2025-12-24,"""8400405ffffffff""",0,0,0,0.0,0.0
2025-12-25,"""8400405ffffffff""",0,0,0,0.0,0.0
2025-12-26,"""8400405ffffffff""",0,0,0,0.0,0.0


## 4. Final Feature Summary

| Column | Role | What It Tells the Model |
|---|---|---|
| `event_count` | **Target** | Activity happening right now |
| `peak_intensity` | **Feature/Target** | Worst severity event recorded |
| `energy_output` | **Feature/Target** | Sustained activity level |
| `lag_1_day` | **Temporal Feature** | What happened yesterday |
| `lag_2_day` | **Temporal Feature** | What happened 2 days ago |
| `lag_7_day` | **Temporal Feature** | Weekly baseline cycle |
| `rolling_mean_3_day` | **Momentum Feature** | Is activity accelerating? |
| `neighbor_pressure_index` | **Spatial Feature** | Crisis encroaching from neighbors? |

In [7]:
print(f'Final ML-Ready Feature Matrix shape: {final_engineered_dataset.shape}')
final_engineered_dataset.schema

Final ML-Ready Feature Matrix shape: (726660, 10)


Schema([('date_bucket', Date),
        ('h3_index', String),
        ('event_count', Int64),
        ('peak_intensity', Float64),
        ('energy_output', Float64),
        ('lag_1_day', Int64),
        ('lag_2_day', Int64),
        ('lag_7_day', Int64),
        ('rolling_mean_3_day', Float64),
        ('neighbor_pressure_index', Float32)])

## 5. Out-of-Time Chronological Validation Split

### Why Random Splits Cause Data Leakage
Random splitting (`train_test_split`) would place Tuesday and Thursday into training,
and Wednesday into test for the SAME hexagon. Since spatial data changes gradually,
the model cheats by seeing correlated temporal neighbors, giving fake high accuracy
that crashes on live data.

### Fix: Strict Chronological Cutoff

| Split | Proportion | Purpose |
|---|---|---|
| **Train** | 60% | Historical foundation for tree weights |
| **Validation** | 20% | Hyperparameter tuning |
| **Test** | 20% | Held-back unseen future evaluation |

In [8]:
unique_dates = final_engineered_dataset['date_bucket'].sort().unique().to_list()
total_days = len(unique_dates)

train_cutoff = unique_dates[int(total_days * 0.60)]
val_cutoff   = unique_dates[int(total_days * 0.80)]

print(f'Total unique calendar days : {total_days}')
print(f'Train Period : {unique_dates[0]}  to  {train_cutoff}')
print(f'Val Period   : {unique_dates[int(total_days*0.60)+1]}  to  {val_cutoff}')
print(f'Test Period  : {unique_dates[int(total_days*0.80)+1]}  to  {unique_dates[-1]}')

Total unique calendar days : 180
Train Period : 2025-12-14  to  2026-04-01
Val Period   : 2026-04-02  to  2026-05-07
Test Period  : 2026-05-08  to  2026-06-11


In [9]:
train_df = final_engineered_dataset.filter(pl.col('date_bucket') <= train_cutoff)
val_df   = final_engineered_dataset.filter((pl.col('date_bucket') > train_cutoff) & (pl.col('date_bucket') <= val_cutoff))
test_df  = final_engineered_dataset.filter(pl.col('date_bucket') > val_cutoff)

total = final_engineered_dataset.height
print(f'Train rows : {train_df.height:,}  ({train_df.height/total*100:.1f}%)')
print(f'Val rows   : {val_df.height:,}  ({val_df.height/total*100:.1f}%)')
print(f'Test rows  : {test_df.height:,}  ({test_df.height/total*100:.1f}%)')

Train rows : 440,033  (60.6%)
Val rows   : 145,332  (20.0%)
Test rows  : 141,295  (19.4%)


In [10]:
feature_cols = ['lag_1_day', 'lag_2_day', 'lag_7_day', 'rolling_mean_3_day', 'neighbor_pressure_index']
target_col   = 'event_count'

X_train = train_df.select(feature_cols).to_numpy()
y_train = train_df.select(target_col).to_numpy().flatten()
X_val   = val_df.select(feature_cols).to_numpy()
y_val   = val_df.select(target_col).to_numpy().flatten()
X_test  = test_df.select(feature_cols).to_numpy()
y_test  = test_df.select(target_col).to_numpy().flatten()

print('Arrays prepared for ML Pipeline:')
print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_val  : {X_val.shape}    |  y_val  : {y_val.shape}')
print(f'X_test : {X_test.shape}   |  y_test : {y_test.shape}')

Arrays prepared for ML Pipeline:
X_train: (440033, 5)  |  y_train: (440033,)
X_val  : (145332, 5)    |  y_val  : (145332,)
X_test : (141295, 5)   |  y_test : (141295,)


## 6. Save ML-Ready Dataset and Splits

In [11]:
final_engineered_dataset.write_csv('ml_ready_feature_matrix.csv')
train_df.write_csv('ml_train_split.csv')
val_df.write_csv('ml_val_split.csv')
test_df.write_csv('ml_test_split.csv')

print('Saved ml_ready_feature_matrix.csv')
print('Saved: ml_train_split.csv | ml_val_split.csv | ml_test_split.csv')
print(f'Total rows: {final_engineered_dataset.height:,}  |  Features: {len(feature_cols)}')

Saved ml_ready_feature_matrix.csv
Saved: ml_train_split.csv | ml_val_split.csv | ml_test_split.csv
Total rows: 726,660  |  Features: 5
